In [1]:
# GPU check
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))


Torch version: 2.5.1+cu121
CUDA available: True
Device: cuda


In [2]:
# imports
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import numpy as np
import torch
import re


In [3]:
# load models
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Clinical classification
CLS_MODEL = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer_cls = AutoTokenizer.from_pretrained(CLS_MODEL)
model_cls = AutoModelForSequenceClassification.from_pretrained(CLS_MODEL).to(DEVICE)
model_cls.eval()

# Embeddings (semantic search, clustering)
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
emb_model = SentenceTransformer(EMB_MODEL, device=DEVICE)

# Retrieval (RAG-style)
RAG_MODEL = "intfloat/e5-base"
rag_model = SentenceTransformer(RAG_MODEL, device=DEVICE)

# Summarisation
SUM_MODEL = "google/pegasus-xsum"
sum_tokenizer = AutoTokenizer.from_pretrained(SUM_MODEL)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(SUM_MODEL).to(DEVICE)
sum_model.eval()


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

C:\Users\marke\projects\data-science\nhs\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\marke\.cache\huggingface\hub\models--emilyalsentzer--Bio_ClinicalBERT. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PegasusForConditionalGeneration(
  (model): PegasusModel(
    (shared): Embedding(96103, 1024, padding_idx=0)
    (encoder): PegasusEncoder(
      (embed_tokens): Embedding(96103, 1024, padding_idx=0)
      (embed_positions): PegasusSinusoidalPositionalEmbedding(512, 1024)
      (layers): ModuleList(
        (0-15): 16 x PegasusEncoderLayer(
          (self_attn): PegasusAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (final_layer_nor

In [4]:
# Sample NHS‑Style Clinical Notes
clinical_notes = [
    "Patient presents with chest pain radiating to left arm, sweating, and shortness of breath. ECG shows ST elevation.",
    "Patient reports mild headache and fatigue. Observations within normal range. Discharged with advice to rest.",
    "Post-operative patient with fever and elevated CRP. Wound site appears red and swollen. Possible infection.",
    "Elderly patient with history of COPD, increased breathlessness over 3 days, using inhaler more frequently.",
    "Patient attended A&E following a fall. X-ray confirms fractured wrist. Plaster applied, follow-up in fracture clinic."
]

pd.DataFrame({"note": clinical_notes})


,note
0,Patient presents with chest pain radiating to ...
1,Patient reports mild headache and fatigue. Obs...
2,Post-operative patient with fever and elevated...
3,"Elderly patient with history of COPD, increase..."
4,Patient attended A&E following a fall. X-ray c...


In [6]:
# cleaning function

def clean_text(text: str) -> str:
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

cleaned_notes = [clean_text(t) for t in clinical_notes]
pd.DataFrame({"cleaned_note": cleaned_notes})

,cleaned_note
0,Patient presents with chest pain radiating to ...
1,Patient reports mild headache and fatigue. Obs...
2,Post-operative patient with fever and elevated...
3,"Elderly patient with history of COPD, increase..."
4,Patient attended A&E following a fall. X-ray c...


In [7]:
# classification (You will replace class_0/class_1 with real harm/severity labels once fine‑tuned.)
def classify_texts(texts):
    inputs = tokenizer_cls(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model_cls(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()

    return probs

probs = classify_texts(cleaned_notes)
pd.DataFrame(probs, columns=["class_0", "class_1"])


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


,class_0,class_1
0,0.514525,0.485475
1,0.591984,0.408016
2,0.550510,0.449490
3,0.235811,0.764189
4,0.563332,0.436668


In [8]:
# embeddings
embeddings = emb_model.encode(cleaned_notes, convert_to_tensor=True)
embeddings.shape

torch.Size([5, 384])

In [10]:
# Similarity Search (MiniLM)
def retrieve_similar(query: str, top_k: int = 3):
    q_emb = emb_model.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, embeddings)[0]
    top_results = torch.topk(scores, k=top_k)

    results = []
    for idx, score in zip(top_results.indices, top_results.values):
        results.append({
            "note": cleaned_notes[idx],
            "score": float(score)
        })
    return pd.DataFrame(results)

query = "Possible infection after surgery"
retrieve_similar(query)


,note,score
0,Post-operative patient with fever and elevated...,0.592312
1,Patient presents with chest pain radiating to ...,0.244488
2,Patient attended A&E following a fall. X-ray c...,0.242944


In [11]:
# RAG‑Style Retrieval (E5‑Base)
rag_embeddings = rag_model.encode(cleaned_notes, convert_to_tensor=True)

def rag_retrieve(query: str, top_k: int = 3):
    q_emb = rag_model.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, rag_embeddings)[0]
    top_results = torch.topk(scores, k=top_k)

    results = []
    for idx, score in zip(top_results.indices, top_results.values):
        results.append({
            "note": cleaned_notes[idx],
            "score": float(score)
        })
    return pd.DataFrame(results)

rag_retrieve("breathlessness and COPD", top_k=3)


,note,score
0,"Elderly patient with history of COPD, increase...",0.899580
1,Patient presents with chest pain radiating to ...,0.819899
2,Patient reports mild headache and fatigue. Obs...,0.787870


In [12]:
# Summarisation (PEGASUS‑XSum)
def summarise(text: str, max_length: int = 64):
    inputs = sum_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="longest"
    ).to(DEVICE)

    with torch.no_grad():
        summary_ids = sum_model.generate(
            **inputs,
            max_length=max_length,
            num_beams=4,
            early_stopping=True
        )

    summary = sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

long_note = (
    "Patient presents with chest pain radiating to left arm, sweating, and shortness of breath. "
    "ECG shows ST elevation consistent with myocardial infarction. "
    "Aspirin and GTN administered in A&E, transferred to cardiology for further management."
)

summarise(long_note)


'We report the case of a 65-year-old man with a history of heart disease who presented with chest pain.'

In [13]:
# full clinical pipeline
def clinical_pipeline(note: str, similarity_query: str = None):
    cleaned = clean_text(note)

    # Classification
    cls_probs = classify_texts([cleaned])[0]

    # Embedding
    emb = emb_model.encode(cleaned, convert_to_tensor=True)

    # Summarisation
    summary = summarise(cleaned)

    # Similarity (MiniLM)
    similar = retrieve_similar(similarity_query) if similarity_query else None

    # RAG retrieval (E5)
    rag_similar = rag_retrieve(similarity_query) if similarity_query else None

    return {
        "cleaned": cleaned,
        "classification_probs": cls_probs,
        "summary": summary,
        "similar": similar,
        "rag_similar": rag_similar
    }

example_note = clinical_notes[2]
result = clinical_pipeline(example_note, similarity_query="infection after surgery")

result


{'cleaned': 'Post-operative patient with fever and elevated CRP. Wound site appears red and swollen. Possible infection.',
 'classification_probs': array([0.5505099 , 0.44949004], dtype=float32),
 'summary': 'A patient with a ruptured Achilles tendon has been admitted to hospital for surgery.',
 'similar':                                                 note     score
 0  Post-operative patient with fever and elevated...  0.555176
 1  Patient attended A&E following a fall. X-ray c...  0.240975
 2  Patient presents with chest pain radiating to ...  0.191833,
 'rag_similar':                                                 note     score
 0  Post-operative patient with fever and elevated...  0.861330
 1  Patient attended A&E following a fall. X-ray c...  0.758739
 2  Patient reports mild headache and fatigue. Obs...  0.752350}

In [14]:
print("CLEANED NOTE:\n", result["cleaned"])
print("\nSUMMARY:\n", result["summary"])
print("\nCLASSIFICATION PROBS:\n", result["classification_probs"])
print("\nSIMILAR NOTES (MiniLM):\n", result["similar"])
print("\nRAG SIMILAR NOTES (E5):\n", result["rag_similar"])


CLEANED NOTE:
 Post-operative patient with fever and elevated CRP. Wound site appears red and swollen. Possible infection.

SUMMARY:
 A patient with a ruptured Achilles tendon has been admitted to hospital for surgery.

CLASSIFICATION PROBS:
 [0.5505099  0.44949004]

SIMILAR NOTES (MiniLM):
                                                 note     score
0  Post-operative patient with fever and elevated...  0.555176
1  Patient attended A&E following a fall. X-ray c...  0.240975
2  Patient presents with chest pain radiating to ...  0.191833

RAG SIMILAR NOTES (E5):
                                                 note     score
0  Post-operative patient with fever and elevated...  0.861330
1  Patient attended A&E following a fall. X-ray c...  0.758739
2  Patient reports mild headache and fatigue. Obs...  0.752350
